In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
#PART B APPROACH 2
import re
import unicodedata

# -----------------------------------
# 1. Load dataset
# -----------------------------------

df = pd.read_csv("/kaggle/input/competitions/shopee-product-matching/train.csv")

print("Dataset shape:", df.shape)
print(train_df.columns.tolist())


# -----------------------------------
# 2. Inspect missing titles
# -----------------------------------

print("Missing titles:", df["title"].isna().sum())


# -----------------------------------
# 3. Keep original title
# -----------------------------------

df["title_original"] = df["title"]


# -----------------------------------
# 4. Preprocessing function
# -----------------------------------

def preprocess_title(text):

    # Missing value
    if pd.isna(text):
        return ""

    # Convert to string
    text = str(text)

    # Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # Lowercase
    text = text.lower()

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove leading/trailing spaces
    text = text.strip()

    return text


# -----------------------------------
# 5. Apply preprocessing
# -----------------------------------

df["title_clean"] = df["title"].apply(preprocess_title)


# -----------------------------------
# 6. Inspect results
# -----------------------------------

print(
    df[
        ["title_original", "title_clean"]
    ].head(20).to_string(index=False)
)


# -----------------------------------
# 7. Check empty titles
# -----------------------------------

empty_count = (
    df["title_clean"]
    .str.strip()
    .eq("")
    .sum()
)

print("Empty titles after preprocessing:", empty_count)

In [ ]:
# ============================================================
# PART B — TF-IDF PRODUCT MATCHING
# Starting point:
#   train_df and val_df already exist
#   train_df["title_clean"] and val_df["title_clean"] exist
#   train_df["label_group"] and val_df["label_group"] exist
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix
)


# ============================================================
# 1. CREATE VALIDATION PAIRS
# ============================================================
# Positive pair:
#   Both products belong to the same label_group
#
# Negative pair:
#   Products belong to different label_groups
#
# We evaluate ONLY on val_df.
# ============================================================

def create_positive_pairs(df):
    pairs = []

    for group_id, group in df.groupby("label_group"):

        indices = group.index.tolist()

        # Generate all unique pairs within the group
        for i in range(len(indices)):
            for j in range(i + 1, len(indices)):

                pairs.append(
                    (indices[i], indices[j], 1)
                )

    return pairs


def create_negative_pairs(df, n_pairs, random_state=42):

    rng = np.random.default_rng(random_state)

    indices = df.index.to_numpy()
    groups = df["label_group"].to_numpy()

    pairs = set()

    while len(pairs) < n_pairs:

        i, j = rng.choice(indices, size=2, replace=False)

        if groups[i] != groups[j]:

            pairs.add((int(i), int(j), 0))

    return list(pairs)


# ------------------------------------------------------------
# Positive pairs
# ------------------------------------------------------------

positive_pairs = create_positive_pairs(val_df)

print("Positive validation pairs:", len(positive_pairs))


# ------------------------------------------------------------
# Negative pairs
# ------------------------------------------------------------

negative_pairs = create_negative_pairs(
    val_df,
    n_pairs=len(positive_pairs),
    random_state=42
)

print("Negative validation pairs:", len(negative_pairs))


# ------------------------------------------------------------
# Combine pairs
# ------------------------------------------------------------

pairs = positive_pairs + negative_pairs

pairs_df = pd.DataFrame(
    pairs,
    columns=["idx_a", "idx_b", "y_true"]
)

pairs_df = pairs_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("\nTotal validation pairs:", len(pairs_df))

print(
    "\nPair distribution:"
)

print(
    pairs_df["y_true"].value_counts()
)


# ============================================================
# 2. TF-IDF MODEL 1 — WORD UNIGRAMS
# ============================================================
# IMPORTANT:
# Fit ONLY on training titles.
# Validation titles are transformed using the same vocabulary.
# ============================================================

word_unigram = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 1),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_word_uni = word_unigram.fit_transform(
    train_df["title_clean"]
)

X_val_word_uni = word_unigram.transform(
    val_df["title_clean"]
)

print("\nWord Unigram TF-IDF")
print("-------------------")
print("Train matrix:", X_train_word_uni.shape)
print("Validation matrix:", X_val_word_uni.shape)
print("Vocabulary size:", len(word_unigram.vocabulary_))


# ============================================================
# 3. TF-IDF MODEL 2 — WORD UNIGRAM + BIGRAM
# ============================================================

word_bigram = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_word_bigram = word_bigram.fit_transform(
    train_df["title_clean"]
)

X_val_word_bigram = word_bigram.transform(
    val_df["title_clean"]
)

print("\nWord 1-2 Gram TF-IDF")
print("--------------------")
print("Train matrix:", X_train_word_bigram.shape)
print("Validation matrix:", X_val_word_bigram.shape)
print("Vocabulary size:", len(word_bigram.vocabulary_))


# ============================================================
# 4. TF-IDF MODEL 3 — CHARACTER N-GRAMS
# ============================================================
# Character n-grams are useful for:
#
# - spelling variations
# - abbreviations
# - product codes
# - model numbers
# - punctuation differences
# - partial word matches
# ============================================================

character_tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2,
    max_features=200000,
    sublinear_tf=True
)

X_train_char = character_tfidf.fit_transform(
    train_df["title_clean"]
)

X_val_char = character_tfidf.transform(
    val_df["title_clean"]
)

print("\nCharacter 3-5 Gram TF-IDF")
print("-------------------------")
print("Train matrix:", X_train_char.shape)
print("Validation matrix:", X_val_char.shape)
print("Vocabulary size:", len(character_tfidf.vocabulary_))


# ============================================================
# 5. CALCULATE PAIRWISE COSINE SIMILARITY
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity


def calculate_pair_scores(X, pairs_df):

    scores = []

    for _, row in pairs_df.iterrows():

        idx_a = int(row["idx_a"])
        idx_b = int(row["idx_b"])

        vector_a = X[idx_a]
        vector_b = X[idx_b]

        score = cosine_similarity(
            vector_a,
            vector_b
        )[0][0]

        scores.append(score)

    return np.array(scores)


# ------------------------------------------------------------
# Word unigram scores
# ------------------------------------------------------------

scores_word_uni = calculate_pair_scores(
    X_val_word_uni,
    pairs_df
)


# ------------------------------------------------------------
# Word unigram + bigram scores
# ------------------------------------------------------------

scores_word_bigram = calculate_pair_scores(
    X_val_word_bigram,
    pairs_df
)


# ------------------------------------------------------------
# Character scores
# ------------------------------------------------------------

scores_char = calculate_pair_scores(
    X_val_char,
    pairs_df
)


# ============================================================
# 6. STORE ALL SCORES
# ============================================================

evaluation_df = pairs_df.copy()

evaluation_df["score_word_unigram"] = scores_word_uni
evaluation_df["score_word_bigram"] = scores_word_bigram
evaluation_df["score_char"] = scores_char

print("\nEvaluation dataframe:")
print(
    evaluation_df.head()
)


# ============================================================
# 7. THRESHOLD EVALUATION FUNCTION
# ============================================================

def evaluate_threshold(
    scores,
    y_true,
    threshold
):

    y_pred = (
        scores >= threshold
    ).astype(int)

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    return {
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy
    }


# ============================================================
# 8. THRESHOLD SWEEP
# ============================================================

thresholds = np.arange(
    0.10,
    0.96,
    0.05
)


def threshold_analysis(
    scores,
    y_true,
    thresholds
):

    results = []

    for threshold in thresholds:

        result = evaluate_threshold(
            scores,
            y_true,
            threshold
        )

        results.append(result)

    return pd.DataFrame(results)


y_true = evaluation_df["y_true"].values


# ------------------------------------------------------------
# Word unigram
# ------------------------------------------------------------

results_word_uni = threshold_analysis(
    scores_word_uni,
    y_true,
    thresholds
)


# ------------------------------------------------------------
# Word bigram
# ------------------------------------------------------------

results_word_bigram = threshold_analysis(
    scores_word_bigram,
    y_true,
    thresholds
)


# ------------------------------------------------------------
# Character
# ------------------------------------------------------------

results_char = threshold_analysis(
    scores_char,
    y_true,
    thresholds
)


# ============================================================
# 9. FIND BEST THRESHOLD FOR EACH MODEL
# ============================================================

best_word_uni = results_word_uni.loc[
    results_word_uni["f1"].idxmax()
]

best_word_bigram = results_word_bigram.loc[
    results_word_bigram["f1"].idxmax()
]

best_char = results_char.loc[
    results_char["f1"].idxmax()
]
g

print("\n======================================")
print("BEST WORD UNIGRAM MODEL")
print("======================================")

print(best_word_uni)


print("\n======================================")
print("BEST WORD BIGRAM MODEL")
print("======================================")

print(best_word_bigram)


print("\n======================================")
print("BEST CHARACTER MODEL")
print("======================================")

print(best_char)


# ============================================================
# 10. HYBRID WORD + CHARACTER TF-IDF
# ============================================================
#
# S_hybrid =
#       alpha * S_word_bigram
#       +
#       (1-alpha) * S_character
#
# This combines:
#
# Word features:
#   semantic-ish lexical phrases
#
# Character features:
#   spelling/model-number robustness
# ============================================================

alpha_values = np.arange(
    0.0,
    1.01,
    0.1
)

hybrid_results = []


for alpha in alpha_values:

    hybrid_scores = (
        alpha * scores_word_bigram
        +
        (1 - alpha) * scores_char
    )

    for threshold in thresholds:

        metrics = evaluate_threshold(
            hybrid_scores,
            y_true,
            threshold
        )

        hybrid_results.append(
            {
                "alpha": alpha,
                **metrics
            }
        )


hybrid_results_df = pd.DataFrame(
    hybrid_results
)


# ============================================================
# 11. FIND BEST HYBRID MODEL
# ============================================================

best_hybrid = hybrid_results_df.loc[
    hybrid_results_df["f1"].idxmax()
]

best_alpha = best_hybrid["alpha"]
best_threshold = best_hybrid["threshold"]


print("\n======================================")
print("BEST HYBRID MODEL")
print("======================================")

print(best_hybrid)


# ============================================================
# 12. CALCULATE FINAL HYBRID SCORES
# ============================================================

final_hybrid_scores = (
    best_alpha * scores_word_bigram
    +
    (1 - best_alpha) * scores_char
)

evaluation_df["score_hybrid"] = (
    final_hybrid_scores
)


# ============================================================
# 13. FINAL MODEL COMPARISON
# ============================================================

comparison = pd.DataFrame({

    "Model": [
        "Word Unigram TF-IDF",
        "Word 1-2 Gram TF-IDF",
        "Character 3-5 Gram TF-IDF",
        "Hybrid Word + Character"
    ],

    "Best Threshold": [
        best_word_uni["threshold"],
        best_word_bigram["threshold"],
        best_char["threshold"],
        best_threshold
    ],

    "Precision": [
        best_word_uni["precision"],
        best_word_bigram["precision"],
        best_char["precision"],
        best_hybrid["precision"]
    ],

    "Recall": [
        best_word_uni["recall"],
        best_word_bigram["recall"],
        best_char["recall"],
        best_hybrid["recall"]
    ],

    "F1": [
        best_word_uni["f1"],
        best_word_bigram["f1"],
        best_char["f1"],
        best_hybrid["f1"]
    ],

    "Accuracy": [
        best_word_uni["accuracy"],
        best_word_bigram["accuracy"],
        best_char["accuracy"],
        best_hybrid["accuracy"]
    ]
})


print("\n======================================")
print("FINAL TF-IDF COMPARISON")
print("======================================")

print(
    comparison.to_string(index=False)
)


# ============================================================
# 14. PLOT PRECISION / RECALL / F1
# ============================================================

plt.figure(figsize=(10, 6))

plt.plot(
    results_word_bigram["threshold"],
    results_word_bigram["precision"],
    marker="o",
    label="Precision"
)

plt.plot(
    results_word_bigram["threshold"],
    results_word_bigram["recall"],
    marker="o",
    label="Recall"
)

plt.plot(
    results_word_bigram["threshold"],
    results_word_bigram["f1"],
    marker="o",
    label="F1"
)

plt.xlabel("Similarity Threshold")
plt.ylabel("Score")
plt.title(
    "Word 1-2 Gram TF-IDF Threshold Analysis"
)

plt.legend()
plt.grid(True)

plt.show()


# ============================================================
# 15. HYBRID THRESHOLD CURVE
# ============================================================

best_alpha_scores = (
    best_alpha * scores_word_bigram
    +
    (1 - best_alpha) * scores_char
)

hybrid_threshold_results = threshold_analysis(
    best_alpha_scores,
    y_true,
    np.arange(0.10, 0.96, 0.01)
)


plt.figure(figsize=(10, 6))

plt.plot(
    hybrid_threshold_results["threshold"],
    hybrid_threshold_results["precision"],
    label="Precision"
)

plt.plot(
    hybrid_threshold_results["threshold"],
    hybrid_threshold_results["recall"],
    label="Recall"
)

plt.plot(
    hybrid_threshold_results["threshold"],
    hybrid_threshold_results["f1"],
    label="F1"
)

plt.axvline(
    best_threshold,
    linestyle="--",
    label=f"Best threshold = {best_threshold:.2f}"
)

plt.xlabel("Similarity Threshold")
plt.ylabel("Score")

plt.title(
    "Hybrid TF-IDF Threshold Sensitivity"
)

plt.legend()
plt.grid(True)

plt.show()


# ============================================================
# 16. FINAL PREDICTIONS
# ============================================================

evaluation_df["y_pred"] = (
    evaluation_df["score_hybrid"]
    >= best_threshold
).astype(int)


# ============================================================
# 17. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    evaluation_df["y_true"],
    evaluation_df["y_pred"]
)

tn, fp, fn, tp = cm.ravel()


print("\n======================================")
print("CONFUSION MATRIX")
print("======================================")

print(cm)

print("\nTrue Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)


# ============================================================
# 18. FINAL METRICS
# ============================================================

final_precision = precision_score(
    evaluation_df["y_true"],
    evaluation_df["y_pred"],
    zero_division=0
)

final_recall = recall_score(
    evaluation_df["y_true"],
    evaluation_df["y_pred"],
    zero_division=0
)

final_f1 = f1_score(
    evaluation_df["y_true"],
    evaluation_df["y_pred"],
    zero_division=0
)

final_accuracy = accuracy_score(
    evaluation_df["y_true"],
    evaluation_df["y_pred"]
)


print("\n======================================")
print("FINAL HYBRID VALIDATION METRICS")
print("======================================")

print(
    f"Precision : {final_precision:.4f}"
)

print(
    f"Recall    : {final_recall:.4f}"
)

print(
    f"F1 Score  : {final_f1:.4f}"
)

print(
    f"Accuracy  : {final_accuracy:.4f}"
)


# ============================================================
# 19. ERROR ANALYSIS
# ============================================================

def classify_result(row):

    if row["y_true"] == 1 and row["y_pred"] == 1:
        return "True Positive"

    elif row["y_true"] == 0 and row["y_pred"] == 0:
        return "True Negative"

    elif row["y_true"] == 0 and row["y_pred"] == 1:
        return "False Positive"

    else:
        return "False Negative"


evaluation_df["result"] = evaluation_df.apply(
    classify_result,
    axis=1
)


# ============================================================
# 20. ADD ACTUAL PRODUCT TITLES
# ============================================================

evaluation_df["title_a"] = evaluation_df[
    "idx_a"
].map(
    val_df["title_clean"]
)

evaluation_df["title_b"] = evaluation_df[
    "idx_b"
].map(
    val_df["title_clean"]
)


# ============================================================
# 21. FALSE POSITIVES
# ============================================================

false_positives = evaluation_df[
    evaluation_df["result"] == "False Positive"
].copy()

false_positives = false_positives.sort_values(
    "score_hybrid",
    ascending=False
)


print("\n======================================")
print("FALSE POSITIVES")
print("======================================")

print(
    false_positives[
        [
            "title_a",
            "title_b",
            "score_word_bigram",
            "score_char",
            "score_hybrid"
        ]
    ].head(20).to_string(index=False)
)


# ============================================================
# 22. FALSE NEGATIVES
# ============================================================

false_negatives = evaluation_df[
    evaluation_df["result"] == "False Negative"
].copy()

false_negatives = false_negatives.sort_values(
    "score_hybrid",
    ascending=True
)


print("\n======================================")
print("FALSE NEGATIVES")
print("======================================")

print(
    false_negatives[
        [
            "title_a",
            "title_b",
            "score_word_bigram",
            "score_char",
            "score_hybrid"
        ]
    ].head(20).to_string(index=False)
)


# ============================================================
# 23. TRUE POSITIVES
# ============================================================

true_positives = evaluation_df[
    evaluation_df["result"] == "True Positive"
].copy()

true_positives = true_positives.sort_values(
    "score_hybrid",
    ascending=False
)


print("\n======================================")
print("TRUE POSITIVES")
print("======================================")

print(
    true_positives[
        [
            "title_a",
            "title_b",
            "score_hybrid"
        ]
    ].head(20).to_string(index=False)
)


# ============================================================
# 24. ERROR COUNTS
# ============================================================

error_counts = (
    evaluation_df["result"]
    .value_counts()
)


print("\n======================================")
print("ERROR ANALYSIS SUMMARY")
print("======================================")

print(error_counts)


# ============================================================
# 25. POSITIVE / NEGATIVE SIMILARITY DISTRIBUTION
# ============================================================

positive_scores = evaluation_df[
    evaluation_df["y_true"] == 1
]["score_hybrid"]

negative_scores = evaluation_df[
    evaluation_df["y_true"] == 0
]["score_hybrid"]


print("\n======================================")
print("SIMILARITY STATISTICS")
print("======================================")

print("\nPositive pairs")

print(
    positive_scores.describe()
)


print("\nNegative pairs")

print(
    negative_scores.describe()
)


# ============================================================
# 26. MEAN SEPARATION
# ============================================================

positive_mean = positive_scores.mean()
negative_mean = negative_scores.mean()

mean_separation = (
    positive_mean - negative_mean
)


print(
    "\nMean separation:",
    mean_separation
)


# ============================================================
# 27. SCORE DISTRIBUTION PLOT
# ============================================================

plt.figure(figsize=(10, 6))

plt.hist(
    positive_scores,
    bins=50,
    alpha=0.6,
    label="Positive pairs"
)

plt.hist(
    negative_scores,
    bins=50,
    alpha=0.6,
    label="Negative pairs"
)

plt.axvline(
    best_threshold,
    linestyle="--",
    label=f"Threshold = {best_threshold:.2f}"
)

plt.xlabel("Hybrid Cosine Similarity")
plt.ylabel("Number of Pairs")

plt.title(
    "Positive vs Negative Pair Similarity"
)

plt.legend()
plt.grid(True)

plt.show()


# ============================================================
# 28. SAVE RESULTS
# ============================================================

comparison.to_csv(
    "tfidf_model_comparison.csv",
    index=False
)

evaluation_df.to_csv(
    "tfidf_pair_evaluation.csv",
    index=False
)

hybrid_threshold_results.to_csv(
    "tfidf_hybrid_threshold_analysis.csv",
    index=False
)

false_positives.to_csv(
    "tfidf_false_positives.csv",
    index=False
)

false_negatives.to_csv(
    "tfidf_false_negatives.csv",
    index=False
)

true_positives.to_csv(
    "tfidf_true_positives.csv",
    index=False
)


statistics_df = pd.DataFrame({

    "Group": [
        "Positive",
        "Negative"
    ],

    "Mean": [
        positive_scores.mean(),
        negative_scores.mean()
    ],

    "Median": [
        positive_scores.median(),
        negative_scores.median()
    ],

    "Std": [
        positive_scores.std(),
        negative_scores.std()
    ],

    "Count": [
        len(positive_scores),
        len(negative_scores)
    ]
})


statistics_df.to_csv(
    "tfidf_similarity_statistics.csv",
    index=False
)


print("\n======================================")
print("FILES SAVED")
print("======================================")

print("tfidf_model_comparison.csv")
print("tfidf_pair_evaluation.csv")
print("tfidf_hybrid_threshold_analysis.csv")
print("tfidf_false_positives.csv")
print("tfidf_false_negatives.csv")
print("tfidf_true_positives.csv")
print("tfidf_similarity_statistics.csv")


# ============================================================
# 29. FINAL SUMMARY
# ============================================================

print("\n======================================")
print("FINAL TF-IDF RESULT")
print("======================================")

print(
    f"Best alpha          : {best_alpha:.2f}"
)

print(
    f"Best threshold      : {best_threshold:.2f}"
)

print(
    f"Validation Precision: {final_precision:.4f}"
)

print(
    f"Validation Recall   : {final_recall:.4f}"
)

print(
    f"Validation F1       : {final_f1:.4f}"
)

print(
    f"Validation Accuracy : {final_accuracy:.4f}"
)